# Data Preparation

### 1. Imports

In [27]:
import pandas as pd

### 2. Lectura inicial

In [28]:
df = pd.read_csv('data/H-AIRosettaMP.csv')

In [29]:
df = df.dropna()
df = df.drop(columns=["task_url", "task_description"])

In [30]:
df.head()

,task_name,language_name,code,target,set
0,Array concatenation,Java,import java.util.Vector;\nimport java.util.Ite...,Ai_generated,Java_from_C++
1,Array length,Java,public class Main {\n public static void ma...,Ai_generated,Java_from_C++
2,Arithmetic/Integer,Java,import java.util.Scanner;\n\npublic class Main...,Ai_generated,Java_from_C++
3,Arithmetic-geometric mean/Calculate Pi,Java,import java.math.BigDecimal;\nimport java.math...,Ai_generated,Java_from_C++
4,Arithmetic/Rational,Java,import java.math.BigInteger;\nimport java.util...,Ai_generated,Java_from_C++


In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 121153 entries, 0 to 121246
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   task_name      121153 non-null  str  
 1   language_name  121153 non-null  str  
 2   code           121153 non-null  str  
 3   target         121153 non-null  str  
 4   set            121153 non-null  str  
dtypes: str(5)
memory usage: 5.5 MB


### 3. Filtrar por lenguajes

In [32]:
# Mantener sólo los que son y vienen de 5 lenguajes
lenguajes = ['Python', 'Java', 'JavaScript', 'C++', 'cpp']
df_filtrado = df[df['language_name'].isin(lenguajes)]
df_filtrado = df_filtrado[df_filtrado['set'].str.endswith(tuple(lenguajes))]

In [33]:
# Filtrar tareas que tienen al menos un ejemplo humano 
tareas_con_humano = df_filtrado[df_filtrado['target'] == 'Human_written'][['task_name', 'language_name']].drop_duplicates()
df_final = df_filtrado.merge(tareas_con_humano, on=['task_name', 'language_name'], how='inner')

In [34]:
print(df_final['target'].value_counts())
print(df_final['language_name'].value_counts())

target
Human_written    7579
Ai_generated     7571
Name: count, dtype: int64
language_name
Python        4109
Java          3909
cpp           3765
JavaScript    3367
Name: count, dtype: int64


### 4. Eliminar impares

In [36]:
cambios_target = df[df['target'] != df['target'].shift()]
cambios_target = cambios_target[['target', 'set']].reset_index()

cambios_target

,index,target,set
0,0,Ai_generated,Java_from_C++
1,612,Human_written,Java_from_C++
2,1224,Ai_generated,Java_from_Python
3,2047,Human_written,Java_from_Python
4,2870,Ai_generated,Java_from_JavaScript
...,...,...,...
175,117654,Human_written,Kotlin_from_Go
176,118447,Ai_generated,Kotlin_from_Ruby
177,119249,Human_written,Kotlin_from_Ruby
178,120051,Ai_generated,Kotlin_from_Rust


In [39]:
conteo = df_final.groupby(['set', 'target']).size().unstack(fill_value=0)
conteo['balanceado'] = conteo['Ai_generated'] == conteo['Human_written']
conteo['diferencia'] = conteo['Ai_generated'] - conteo['Human_written']
conteo['ratio'] = conteo['Ai_generated'] / (conteo['Human_written'] + 1e-9)

conteo

target,Ai_generated,Human_written,balanceado,diferencia,ratio
set,,,,,
C++_from_Java,656,656,True,0,1.000000
C++_from_JavaScript,479,480,False,-1,0.997917
C++_from_Python,747,747,True,0,1.000000
JavaScript_from_C++,480,483,False,-3,0.993789
JavaScript_from_Java,568,568,True,0,1.000000
JavaScript_from_Python,634,634,True,0,1.000000
Java_from_C++,612,612,True,0,1.000000
Java_from_JavaScript,520,521,False,-1,0.998081
Java_from_Python,822,822,True,0,1.000000


In [41]:
df_balanced = []

for (set, task), group in df_final.groupby(['set', 'task_name']):
    human = group[group['target'] == 'Human_written']
    ai = group[group['target'] == 'Ai_generated']
    
    # Solo entra si hay al menos uno de ambos
    if len(human) > 0 and len(ai) > 0:
        n = min(len(human), len(ai))
        
        df_balanced.append(pd.concat([
            human.sample(n, random_state=42),
            ai.sample(n, random_state=42)
        ]))

df_balanced = pd.concat(df_balanced).reset_index(drop=True)

In [42]:
df_balanced.info()

<class 'pandas.DataFrame'>
RangeIndex: 15142 entries, 0 to 15141
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   task_name      15142 non-null  str  
 1   language_name  15142 non-null  str  
 2   code           15142 non-null  str  
 3   target         15142 non-null  str  
 4   set            15142 non-null  str  
dtypes: str(5)
memory usage: 591.6 KB


In [44]:
conteo_bal = df_balanced.groupby(['set', 'target']).size().unstack(fill_value=0)
conteo_bal['balanceado'] = conteo_bal['Ai_generated'] == conteo_bal['Human_written']
conteo_bal['diferencia'] = conteo_bal['Ai_generated'] - conteo_bal['Human_written']
conteo_bal['ratio'] = conteo_bal['Ai_generated'] / (conteo_bal['Human_written'] + 1e-9)

conteo_bal

target,Ai_generated,Human_written,balanceado,diferencia,ratio
set,,,,,
C++_from_Java,656,656,True,0,1.0
C++_from_JavaScript,479,479,True,0,1.0
C++_from_Python,747,747,True,0,1.0
JavaScript_from_C++,480,480,True,0,1.0
JavaScript_from_Java,568,568,True,0,1.0
JavaScript_from_Python,634,634,True,0,1.0
Java_from_C++,612,612,True,0,1.0
Java_from_JavaScript,520,520,True,0,1.0
Java_from_Python,822,822,True,0,1.0


### 5. Crear pares

In [48]:
humans = df_final[df_final['target'] == 'Human_written']
ia = df_final[df_final['target'] == 'Ai_generated']

print(humans['language_name'].value_counts())
print(ia['language_name'].value_counts())

language_name
Python        957
Java          920
cpp           875
JavaScript    668
Name: count, dtype: int64
language_name
Python        2014
Java          1948
cpp           1872
JavaScript    1622
Name: count, dtype: int64


In [49]:
ia2 = ia[ia['language_name'] == 'Python']
ia2.head()

,task_name,language_name,code,target,set
5615,Array concatenation,Python,"def concat(arr1, arr2):\n res = [None] * (l...",Ai_generated,Python_from_Java
5616,Arithmetic/Integer,Python,import java.util.Scanner;\n \npublic class Flo...,Ai_generated,Python_from_Java
5617,Arithmetic-geometric mean/Calculate Pi,Python,import math\nfrom decimal import *\n\ngetconte...,Ai_generated,Python_from_Java
5618,Arithmetic-geometric mean,Python,# Arithmetic-Geometric Mean of 1 & 1/sqrt(2)\n...,Ai_generated,Python_from_Java
5619,Arithmetic/Rational,Python,from fractions import Fraction\n\nMAX_NUM = 1 ...,Ai_generated,Python_from_Java


In [50]:
df_final.to_csv('data/filtered_H-AIRosettaMP.csv', index=False)

In [51]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 10876 entries, 0 to 10875
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   task_name      10876 non-null  str  
 1   language_name  10876 non-null  str  
 2   code           10876 non-null  str  
 3   target         10876 non-null  str  
 4   set            10876 non-null  str  
dtypes: str(5)
memory usage: 425.0 KB
